In [ ]:
from itertools import permutations
import pickle
import socket
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


np.random.seed(0)
torch.manual_seed(0)

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from fig_utils.transformed_rnn import transformed_rnn
from vi_rnn.data_utils import make_all_trials
from vi_rnn.saving import CPU_Unpickler, load_model

from fig_utils.decoding import (
    eval_gen_decoder,
    logistic_from_eff_weights,
    print_acc_summary,
)
from fig_utils.perturbation import (
    build_perturbation_table,
    generate_w_perturb_x,
    latent_mean,
    precompute_perturb_geometry,
    run_null_baseline,
    run_perturbation_sweep,
    position_latent_indices,
    admissible_targets,
)
from fig_utils.plots import (
    plot_decision_distributions,
    plot_fraction_changed_from_summary,
    plot_perturbation_latent_snapshots,
)

In [ ]:
hostname = socket.gethostname()
print("hostname:", hostname)

if hostname == "MatthijsDesktop":
    out_dir = Path("/home/matthijs/swm_rnn/final_models/macaque")
    path = "/home/matthijs/swm_rnn/data/"
else:
    out_dir = Path("/Users/matthijs/swm_rnn_cl/final_models/macaque")
    path = str(Path.cwd().parent / "data") + "/"

model_dirs = [
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_28_T_05_34_01",
    "SWM_low_rank_one_to_one_dim_z_64_date_2026_05_01_T_22_03_56",
    "SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_18_03_12",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_17_02_13",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_30_36",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_26_38",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_27_25",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_30_22",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_20_35",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_13_05",
]

In [ ]:
df_basii = pickle.load(open("../data/processed/df_basii.pkl", "rb"))
df_decoding = pickle.load(open("../data/processed/df_decoding.pkl", "rb"))

In [ ]:
# --- controls ---

# --- task ---
n_pos = 3
n_stim = 6
trial_dur = 5.5

# --- data ---
n_duplications = 10

# --- windows ---
bins_before = 4
bins_after = 1

# --- perturb ---
perturb_at_t = 50  # bin at which x-space perturbation is applied
t_snapshot = 60  # bin shown in before/after latent panels
goal_amp = 1.0
optogen_pct = 5

# --- style ---
cmap = mpl.colors.ListedColormap(sns.color_palette("husl", n_colors=n_stim))

# --- run ---
n_pcs_time = 2
max_models = 10
noise_scale = 1.0
run = False

# --- misc ---
n_repeated_trials_plots = 120
n_plots_per_model = 1

In [ ]:
task_params_file = str(out_dir) + "/" + model_dirs[0] + "_task_params.pkl"
with open(task_params_file, "rb") as f:
    task_params = CPU_Unpickler(f).load()
bin_size = task_params["bin_size"]

# 120 unique stimulus sequences (one trial each)
u, _, labels, delay_ends = make_all_trials(
    task_params,
    dur=trial_dur,
    n_stim=n_stim,
    n_pos=n_pos,
    cue_dur=None,
    bin_size=bin_size,
    interval_dur="mean",
    delay_dur="mean",
)

# u_gen: duplicated trials, use for condition means, goals, demo plots
u_gen = np.concatenate([u] * n_duplications, axis=0)
labels_gen = np.concatenate([labels] * n_duplications, axis=0)
delay_ends_gen = np.concatenate([delay_ends] * n_duplications, axis=0)


delay_end_ref = int(np.round(delay_ends.mean()))
perturb_table = build_perturbation_table(labels, n_pos, n_stim)

In [ ]:
if run:
    rows = []

    for i, model_dir_name in enumerate(model_dirs[:max_models]):
        model_dir = out_dir / Path(model_dir_name)
        name = model_dir.name

        vae, training_params, task_params = load_model(
            str(model_dir), load_encoder=True, backward_compat=False
        )
        task_params["path"] = path

        print("\n ------------- ")
        print("model name:", name)
        print("macaque:", task_params["sessions"][0][5:10])
        print(f"Processing model {i + 1} of {min(len(model_dirs), max_models)}")

        row_b = df_basii.loc[df_basii["name"] == name]
        if row_b.empty:
            print(f"skip {name}: not in df_basii")
            continue
        row_d = df_decoding.loc[df_decoding["name"] == name]
        if row_d.empty:
            print(f"skip {name}: not in df_decoding")
            continue

        # initialise RNN
        rnn_orth = transformed_rnn(
            vae, row_b["A_comb_np"].values[0], row_b["b_comb_np"].values[0]
        )

        # initialise decoder
        w_eff = row_d["w_eff"].values[0]
        b_eff = row_d["b_eff"].values[0]
        response_times = row_d["mean_response_onsets"].values[0]
        logistic_regression_model = logistic_from_eff_weights(
            w_eff, b_eff, n_classes=n_stim
        )

        # evaluate decoder on generated trials
        z_gen = rnn_orth.simulate(u_gen, noise_scale=noise_scale)
        acc_gen = eval_gen_decoder(
            z_gen,
            labels_gen,
            delay_ends_gen,
            logistic_regression_model,
            response_times,
            n_pos,
            bins_before,
            bins_after,
        )
        print("Gen decoder accuracy:", acc_gen)

        # compute goal means for each position
        goal_means = latent_mean(
            z_gen, labels_gen, perturb_at_t, n_pcs_time, n_pos, n_stim
        )
        pert_space = precompute_perturb_geometry(rnn_orth, n_pcs_time, n_pos)

        # generate plots for one example random perturbation
        for _ in range(n_plots_per_model):
            pos = np.random.randint(n_pos)
            seq_i = np.random.randint(len(labels_gen))
            valid_target = admissible_targets(labels_gen[seq_i], pos, n_pos, n_stim)
            target = np.random.choice(valid_target)
            labels_highlight = labels_gen[seq_i]
            source = int(labels_highlight[pos])
            print(
                f"  demo {seq_i}: sequence {list(labels_highlight)}, "
                f"perturb rank {pos} (stim {source} -> {target})"
            )
            pert_dir = pert_space[pos]

            # many repeated trials of the same condition
            u_src = np.repeat(u_gen[seq_i : seq_i + 1], n_repeated_trials_plots, axis=0)
            z_target_mean = goal_means[pos, target]

            # generate unperturbed and perturbed latent trajectories
            Z_unperturbed = generate_w_perturb_x(
                rnn_orth, u=u_src, noise_scale=noise_scale
            )
            Z_perturbed = generate_w_perturb_x(
                rnn_orth,
                u=u_src,
                noise_scale=noise_scale,
                perturb_at_t=perturb_at_t,
                perturb_weights=pert_dir,
                perturb_inds=list(position_latent_indices(n_pcs_time, pos)),
                goal=z_target_mean,
                goal_amp=goal_amp,
                optogen=0,
            )
            # generate plots
            plot_perturbation_latent_snapshots(
                Z_unperturbed,
                Z_perturbed,
                z_gen,
                labels_gen,
                labels_highlight,
                t_bin=t_snapshot,
                n_pcs_time=n_pcs_time,
                n_pos=n_pos,
                cmap=cmap,
                bin_size=bin_size,
                dpi=300,
                show=True,
            )

            # get response distributions
            accs, responses, valid = eval_gen_decoder(
                Z_unperturbed,
                labels_gen,
                delay_ends_gen,
                logistic_regression_model,
                response_times,
                n_pos,
                bins_before,
                bins_after,
                return_preds=True,
            )
            responses = np.apply_along_axis(
                lambda x: np.bincount(x, minlength=n_stim).astype(float),
                axis=0,
                arr=responses,
            ).T
            responses /= responses.sum(axis=1, keepdims=True)

            accs, responses_perturbed, valid = eval_gen_decoder(
                Z_perturbed,
                labels_gen,
                delay_ends_gen,
                logistic_regression_model,
                response_times,
                n_pos,
                bins_before,
                bins_after,
                return_preds=True,
            )
            responses_perturbed = np.apply_along_axis(
                lambda x: np.bincount(x, minlength=n_stim).astype(float),
                axis=0,
                arr=responses_perturbed,
            ).T
            responses_perturbed /= responses_perturbed.sum(axis=1, keepdims=True)

            # plot response distributions
            plot_decision_distributions(
                responses,
                responses_perturbed,
                labels_highlight,
                target_cond=target,
                perturb_pos=pos,
                cmap=cmap,
                n_stim=n_stim,
                dpi=300,
                show=True,
            )

        ## -----
        ## now do systematic run
        ## -----

        # run baseline
        noise_null = run_null_baseline(
            rnn_orth,
            logistic_regression_model,
            u,
            labels,
            delay_ends,
            response_times,
            n_pos,
            noise_scale,
            bins_before,
            bins_after,
        )
        print("noise-null acc:", noise_null["acc"])

        sweep_regular = run_perturbation_sweep(
            rnn_orth,
            logistic_regression_model,
            u,
            labels,
            delay_ends,
            response_times,
            perturb_table,
            goal_means,
            pert_space,
            n_pos=n_pos,
            n_pcs_time=n_pcs_time,
            noise_scale=noise_scale,
            perturb_at_t=perturb_at_t,
            goal_amp=goal_amp,
            optogen=0,
            bins_before=bins_before,
            bins_after=bins_after,
        )
        print("regular accucacy all trials: ", sweep_regular["acc_all"])
        print("regular pert acc:", sweep_regular["acc"])
        sweep_optogen = run_perturbation_sweep(
            rnn_orth,
            logistic_regression_model,
            u,
            labels,
            delay_ends,
            response_times,
            perturb_table,
            goal_means,
            pert_space,
            n_pos=n_pos,
            n_pcs_time=n_pcs_time,
            noise_scale=noise_scale,
            perturb_at_t=perturb_at_t,
            goal_amp=goal_amp,
            optogen=optogen_pct,
            bins_before=bins_before,
            bins_after=bins_after,
        )
        print("optogen accucacy all trials: ", sweep_optogen["acc_all"])
        print("optogen pert acc:", sweep_optogen["acc"])

        rows.append(
            {
                "name": name,
                "macaque": task_params["sessions"][0][5:10],
                "gen_acc": acc_gen,
                "noise_null_acc": noise_null["acc"],
                "noise_null_fraction_changed": noise_null["fraction_changed"],
                "perturb_target_acc_regular": sweep_regular["acc"],
                "perturb_target_acc_optogen": sweep_optogen["acc"],
                "fraction_changed_regular": sweep_regular["fraction_changed"],
                "fraction_changed_optogen": sweep_optogen["fraction_changed"],
                "perturb_target_acc_all_regular": sweep_regular["acc_all"],
                "perturb_target_acc_all_optogen": sweep_optogen["acc_all"],
            }
        )

In [ ]:
if run:
    df = pd.DataFrame(rows)
    df.to_pickle("../data/processed/df_perturbation.pkl")
else:
    df = pd.read_pickle("../data/processed/df_perturbation.pkl")

In [ ]:
# print #gen_acc, noise_null_acc, , perturb_target_acc_regular, perturb_target_acc_optogen, , , perturb_target_acc_all_regular, perturb_target_acc_all_optogen

print_acc_summary(df, "Gen trials (u_gen)", "gen_acc", n_pos=n_pos)
print_acc_summary(df, "baseline (u)", "noise_null_acc", n_pos=n_pos)
print_acc_summary(
    df, "all trials regular (u)", "perturb_target_acc_all_regular", n_pos=n_pos
)
print_acc_summary(
    df, "all trials optogen (u)", "perturb_target_acc_all_optogen", n_pos=n_pos
)
print_acc_summary(
    df, "perturb regular (u_src)", "perturb_target_acc_regular", n_pos=n_pos
)
print_acc_summary(
    df, "perturb optogen (u_src)", "perturb_target_acc_optogen", n_pos=n_pos
)

In [ ]:
import importlib
import fig_utils.plots as plots

importlib.reload(plots)
from fig_utils.plots import plot_fraction_changed_from_summary

In [ ]:
plot_fraction_changed_from_summary(
    df,
    dpi=300,
    save_path="../paper_figures/fraction_changed.pdf",
    show=True,
    # box_w=.7,
    # box_h=.7,
    # panel_gap_x=0.19,
);